# MLB API Data Ingestion

This notebook explores and extracts data from the ESPN MLB API. The goal is to flatten the JSON structures into Pandas DataFrames, ensuring the schemas are correct before moving them to production `.py` files for cron job ingestion.

In [1]:
import pandas as pd
import httpx
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

## 1. Dimensions: Seasons & Season Types

We start at the highest level index to gather all chronological data. This sets up our primary keys (`season_year`) and season type categories (Regular Season, Postseason, etc.). Note the use of `limit=250` to ensure we grab all 150+ years of history in one request. We use `ThreadPoolExecutor` to parallelize the individual season detail requests for speed.

*Note: All dates are explicitly converted to UTC using `pd.to_datetime(..., utc=True)`.*

In [2]:
seasons_index_url = "https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/seasons"
params = {'lang': 'en', 'region': 'us', 'limit': 250}

seasons_records = []
season_types_records = []

def fetch_data(client, url, max_retries=3):
    import time
    for attempt in range(max_retries):
        resp = client.get(url)
        if resp.status_code == 403 or resp.status_code == 429:
            # Rate limited. Wait and backoff.
            wait_time = (2 ** attempt) + 1  # 2s, 5s, 9s...
            print(f"Rate limited (403/429) on {url}. Retrying in {wait_time}s...")
            time.sleep(wait_time)
            continue
        resp.raise_for_status()
        return resp.json()
    raise Exception(f"Failed to fetch {url} after {max_retries} attempts due to rate limiting.")

print(f"Fetching master list from {seasons_index_url}...")
start_time = time.time()

# Use an increased connection limit to support high concurrency
with httpx.Client(limits=httpx.Limits(max_connections=100)) as client:
    index_resp = client.get(seasons_index_url, params=params)
    index_resp.raise_for_status()
    index_data = index_resp.json()
    
    urls = [item['$ref'] for item in index_data.get('items', [])]
    print(f"Found {len(urls)} seasons to process. Starting parallel fetch...")
    
    with ThreadPoolExecutor(max_workers=50) as executor:
        future_to_url = {executor.submit(fetch_data, client, url): url for url in urls}
        
        for future in as_completed(future_to_url):
            url = future_to_url[future]
            try:
                season_data = future.result()
                
                # A) Build the Main Season Record (Forcing UTC for all dates)
                seasons_records.append({
                    'season_year': int(season_data.get('year')),
                    'start_date': pd.to_datetime(season_data.get('startDate'), utc=True).replace(tzinfo=None),
                    'end_date': pd.to_datetime(season_data.get('endDate'), utc=True).replace(tzinfo=None),
                    'display_name': season_data.get('displayName')
                })
                
                # B) Extract the "Season Types" (Preseason, Regular, Postseason)
                if 'types' in season_data and 'items' in season_data['types']:
                    for s_type in season_data['types']['items']:
                        season_types_records.append({
                            'season_year': int(season_data.get('year')), 
                            'type_id': int(s_type.get('id')),
                            'name': s_type.get('name'),
                            'abbreviation': s_type.get('abbreviation'),
                            'start_date': pd.to_datetime(s_type.get('startDate'), utc=True).replace(tzinfo=None) if s_type.get('startDate') else None,
                            'end_date': pd.to_datetime(s_type.get('endDate'), utc=True).replace(tzinfo=None) if s_type.get('endDate') else None,
                            'has_standings': s_type.get('hasStandings', False)
                        })
                        
            except Exception as e:
                print(f"Error fetching detail for {url}: {e}")

df_seasons = pd.DataFrame(seasons_records)
# FILTER FOR DEMO: Only keep 2019 to current
df_seasons = df_seasons[df_seasons['season_year'] >= 2019]
df_seasons = df_seasons.sort_values('season_year', ascending=False).reset_index(drop=True)
# Set season_year as the explicit primary key column
df_seasons.set_index('season_year', inplace=True, drop=False)
df_season_types = pd.DataFrame(season_types_records).sort_values(['season_year', 'type_id'], ascending=[False, True]).reset_index(drop=True)
# Create a composite primary key for season_types
df_season_types['season_type_id'] = df_season_types['season_year'].astype(str) + '_' + df_season_types['type_id'].astype(str)
df_season_types.set_index('season_type_id', inplace=True, drop=False)

end_time = time.time()
print(f"\nSeasons finished in {end_time - start_time:.2f} seconds!")

display(df_seasons.head(2))



Fetching master list from https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/seasons...
Found 151 seasons to process. Starting parallel fetch...

Seasons finished in 0.87 seconds!


,season_year,start_date,end_date,display_name
season_year,,,,
2026,2026,2026-02-19 08:00:00,2026-11-12 07:59:00,2026
2025,2025,2025-02-15 08:00:00,2025-12-11 07:59:00,2025


## 2. Entities: Teams per Season

Instead of grabbing just the 30 active current teams, we fetch teams *per season*. This ensures historically accurate data (e.g., getting the "Montreal Expos" instead of the "Washington Nationals" when querying data for 1990).

We do this in two extremely fast parallel passes:
1. Hit `.../seasons/{year}/teams` for all years to get the specific `$ref` URLs.
2. Hit the ~4,000+ `$ref` URLs concurrently to build the massive `season_teams` table.

In [3]:
print(f"Building URLs for {len(df_seasons)} seasons...")
start_time = time.time()

# Construct the endpoint for every season we found earlier
season_team_urls = [
    f"https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/seasons/{year}/teams?lang=en&region=us&limit=150"
    for year in df_seasons['season_year']
]

team_detail_refs = []
teams_records = []

with httpx.Client(limits=httpx.Limits(max_connections=100)) as client:
    # --- PASS 1: Get the $ref URLs for every team in every season ---
    print("Pass 1: Fetching list of team references per season...")
    with ThreadPoolExecutor(max_workers=50) as executor:
        future_to_url = {executor.submit(fetch_data, client, url): url for url in season_team_urls}
        for future in as_completed(future_to_url):
            try:
                data = future.result()
                if 'items' in data:
                    team_detail_refs.extend([item['$ref'] for item in data['items']])
            except Exception as e:
                pass # Can add logging here if needed
                
    print(f"Pass 1 Complete. Found {len(team_detail_refs)} historical team records.")
    print("Pass 2: Fetching individual team details concurrently...")
    
    # --- PASS 2: Fetch the actual team details ---
    with ThreadPoolExecutor(max_workers=100) as executor:
        future_to_url = {executor.submit(fetch_data, client, url): url for url in team_detail_refs}
        
        for i, future in enumerate(as_completed(future_to_url)):
            if i > 0 and i % 500 == 0:
                print(f"  ...processed {i}/{len(team_detail_refs)} records...")
                
            try:
                team_data = future.result()
                url = future_to_url[future]
                
                # Extract year directly from the HATEOAS URL to guarantee mapping
                # Example: .../seasons/1990/teams/20?lang=en&region=us
                season_year = int(url.split('/seasons/')[1].split('/')[0])
                
                teams_records.append({
                    'season_year': season_year,
                    'team_id': int(team_data.get('id')),
                    'uid': team_data.get('uid'),
                    'location': team_data.get('location'),
                    'name': team_data.get('name'),
                    'abbreviation': team_data.get('abbreviation'),
                    'display_name': team_data.get('displayName'),
                    'color': team_data.get('color'),
                    'alternate_color': team_data.get('alternateColor'),
                    'is_active': team_data.get('isActive'),
                    # Extract underlying Franchise/Venue IDs
                    'franchise_id': int(team_data.get('franchise', {}).get('$ref', '').split('/franchises/')[1].split('?')[0]) if 'franchise' in team_data else None,
                    'venue_id': int(team_data.get('venue', {}).get('$ref', '').split('/venues/')[1].split('?')[0]) if 'venue' in team_data else None,
                    # Extract the group/division ID from the groups $ref link
                    'group_id': int(team_data.get('groups', {}).get('$ref', '').split('/groups/')[1].split('?')[0]) if 'groups' in team_data else None,
                })
                
            except Exception as e:
                pass # Can add logging here if needed

df_season_teams = pd.DataFrame(teams_records).sort_values(['season_year', 'team_id'], ascending=[False, True]).reset_index(drop=True)

# Create a unique primary key for a team in a specific season
df_season_teams['season_team_id'] = df_season_teams['season_year'].astype(str) + '_' + df_season_teams['team_id'].astype(str)
df_season_teams.set_index('season_team_id', inplace=True, drop=False)

end_time = time.time()
print(f"\nSeason-Teams finished in {end_time - start_time:.2f} seconds!")

# Let's show 1990 to prove we captured historical names like the Montreal Expos!


Building URLs for 8 seasons...
Pass 1: Fetching list of team references per season...
Pass 1 Complete. Found 240 historical team records.
Pass 2: Fetching individual team details concurrently...

Season-Teams finished in 0.86 seconds!


## 3. Entities: Groups (Leagues & Divisions)

MLB teams belong to Divisions, which belong to Leagues. The ESPN API tracks this in the `/groups` endpoint. This maps nicely to the `group_id` we just added to the `season_teams` table.

In [4]:
print(f"Building URLs for {len(df_seasons)} seasons to fetch groups...")
start_time = time.time()

# Groups are attached to the 'Regular Season' type (type 2)
season_group_urls = [
    f"https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/seasons/{year}/types/2/groups?lang=en&region=us"
    for year in df_seasons['season_year']
]

group_detail_refs = []
groups_records = []

with httpx.Client(limits=httpx.Limits(max_connections=100)) as client:
    # --- PASS 1: Get the League URLs (Parents) ---
    with ThreadPoolExecutor(max_workers=50) as executor:
        future_to_url = {executor.submit(fetch_data, client, url): url for url in season_group_urls}
        for future in as_completed(future_to_url):
            try:
                data = future.result()
                if 'items' in data:
                    group_detail_refs.extend([item['$ref'] for item in data['items']])
            except Exception:
                pass
                
    # --- PASS 2: Get Division URLs (Children of Leagues) ---
    child_refs = []
    with ThreadPoolExecutor(max_workers=50) as executor:
        future_to_url = {executor.submit(fetch_data, client, url.replace('?lang', '/children?lang')): url for url in group_detail_refs}
        for future in as_completed(future_to_url):
            try:
                data = future.result()
                if 'items' in data:
                    child_refs.extend([item['$ref'] for item in data['items']])
            except Exception:
                pass
                
    # Combine both Leagues and Divisions to fetch their exact details
    all_group_refs = list(set(group_detail_refs + child_refs))
    print(f"Found {len(all_group_refs)} historical group/division records. Fetching details...")
    
    # --- PASS 3: Fetch the actual group details ---
    with ThreadPoolExecutor(max_workers=100) as executor:
        future_to_url = {executor.submit(fetch_data, client, url): url for url in all_group_refs}
        
        for future in as_completed(future_to_url):
            try:
                g_data = future.result()
                url = future_to_url[future]
                
                # Extract year safely
                season_year = int(url.split('/seasons/')[1].split('/')[0])
                
                parent_id = None
                if 'parent' in g_data:
                     parent_id = int(g_data['parent']['$ref'].split('/groups/')[1].split('?')[0])
                
                groups_records.append({
                    'season_year': season_year,
                    'group_id': int(g_data.get('id')),
                    'name': g_data.get('name'),
                    'abbreviation': g_data.get('abbreviation'),
                    'is_conference': g_data.get('isConference', False),
                    'parent_group_id': parent_id
                })
                
            except Exception:
                pass

df_groups = pd.DataFrame(groups_records)
# Create composite primary key for groups (Year + GroupID)
df_groups['season_group_id'] = df_groups['season_year'].astype(str) + '_' + df_groups['group_id'].astype(str)
df_groups.set_index('season_group_id', inplace=True, drop=False)
df_groups = df_groups.sort_values(['season_year', 'group_id'], ascending=[False, True]).reset_index(drop=True)

end_time = time.time()
print(f"\nGroups finished in {end_time - start_time:.2f} seconds!")

display(df_groups[df_groups['season_year'] == 2024].head(10))

Building URLs for 8 seasons to fetch groups...
Found 64 historical group/division records. Fetching details...

Groups finished in 0.79 seconds!


,season_year,group_id,name,abbreviation,is_conference,parent_group_id,season_group_id
16,2024,1,American League East,ALE,False,7,2024_1
17,2024,2,American League Central,ALC,False,7,2024_2
18,2024,3,American League West,ALW,False,7,2024_3
19,2024,4,National League East,NLE,False,8,2024_4
20,2024,5,National League Central,NLC,False,8,2024_5
21,2024,6,National League West,NLW,False,8,2024_6
22,2024,7,American League,AL,True,9,2024_7
23,2024,8,National League,NL,True,9,2024_8


## 4. Dimension: Positions

Before we grab the players, we need to map the Baseball Positions (e.g. `1` = Pitcher, `2` = Catcher). This is a small but critical dimension table.

In [5]:
positions_url = "https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/positions?limit=50"
positions_records = []

print(f"Fetching Positions from {positions_url}...")
with httpx.Client() as client:
    pos_resp = client.get(positions_url)
    pos_resp.raise_for_status()
    pos_data = pos_resp.json()
    
    pos_urls = [item['$ref'] for item in pos_data.get('items', [])]
    
    # It's a small list (~23 items), so standard sequential fetch is fine
    for url in pos_urls:
        try:
            p_data = client.get(url).json()
            positions_records.append({
                'position_id': int(p_data.get('id')),
                'name': p_data.get('name'),
                'display_name': p_data.get('displayName'),
                'abbreviation': p_data.get('abbreviation'),
                'leaf': p_data.get('leaf', True)
            })
        except Exception:
            pass

df_positions = pd.DataFrame(positions_records)
df_positions = df_positions.sort_values('position_id').reset_index(drop=True)
df_positions.set_index('position_id', inplace=True, drop=False)


Fetching Positions from https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/positions?limit=50...


## 5. Entities: Athletes (Global Index)

ESPN maintains a global index of all ~38,000 baseball players in their database at `/athletes`. 
We will fetch this index using `limit=250` to get all the pages, and then batch process the player biographies.

In [6]:
athletes_index_url = "https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/athletes"
athlete_refs = []

print(f"Fetching global athletes index from {athletes_index_url}...")

with httpx.Client() as client:
    # Hit page 1 to get the total pageCount
    initial_resp = fetch_data(client, f"{athletes_index_url}?limit=1000&page=1")
    page_count = initial_resp.get('pageCount', 1)
    print(f"Found {initial_resp.get('count')} total athletes across {page_count} pages.")
    
    # Extract from page 1
    if 'items' in initial_resp:
        athlete_refs.extend([item['$ref'] for item in initial_resp['items']])
        
    # Fetch the rest of the pages (2 through page_count)
    page_urls = [f"{athletes_index_url}?limit=1000&page={p}" for p in range(2, page_count + 1)]
    
    with ThreadPoolExecutor(max_workers=20) as executor:
        future_to_url = {executor.submit(fetch_data, client, url): url for url in page_urls}
        for i, future in enumerate(as_completed(future_to_url)):
            if i > 0 and i % 25 == 0:
                print(f"  ...fetched {i}/{len(page_urls)} index pages...")
            try:
                data = future.result()
                if 'items' in data:
                    athlete_refs.extend([item['$ref'] for item in data['items']])
            except Exception:
                pass

# Ensure uniqueness just in case
athlete_ref_list = sorted(list(set(athlete_refs)))
print(f"\nSuccessfully extracted {len(athlete_ref_list)} unique athlete URLs.")

if 'global_athlete_records' not in locals():
    global_athlete_records = []
if 'last_processed_idx' not in locals():
    last_processed_idx = 0


Fetching global athletes index from https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/athletes...
Found 37682 total athletes across 38 pages.
  ...fetched 25/37 index pages...

Successfully extracted 37682 unique athlete URLs.


In [32]:
# --- MANUAL BATCH EXECUTION ---
# Run this cell multiple times! It will process BATCH_SIZE athletes,
# save them to the global list, and remember where it left off.
BATCH_SIZE = 1500

start_idx = last_processed_idx
end_idx = min(start_idx + BATCH_SIZE, len(athlete_ref_list))

if start_idx >= len(athlete_ref_list):
    print("All athletes have already been processed! You can run the final DB cell now.")
else:
    batch_refs = athlete_ref_list[start_idx:end_idx]
    print(f"Fetching bios for batch: {start_idx} to {end_idx - 1}...")
    start_time = time.time()
    
    with httpx.Client(limits=httpx.Limits(max_connections=150)) as client:
        with ThreadPoolExecutor(max_workers=100) as executor: 
            future_to_url = {executor.submit(fetch_data, client, url): url for url in batch_refs}
            
            for i, future in enumerate(as_completed(future_to_url)):
                if i > 0 and i % 100 == 0:
                    print(f"  ...processed {i}/{len(batch_refs)} in this batch...")
                try:
                    a_data = future.result()
                    global_athlete_records.append({
                        'athlete_id': int(a_data.get('id')),
                        'uid': a_data.get('uid'),
                        'first_name': a_data.get('firstName'),
                        'last_name': a_data.get('lastName'),
                        'full_name': a_data.get('fullName'),
                        'display_name': a_data.get('displayName'),
                        'weight': a_data.get('weight'),
                        'height': a_data.get('height'),
                        'age': a_data.get('age'),
                        'date_of_birth': pd.to_datetime(a_data.get('dateOfBirth'), utc=True).replace(tzinfo=None) if a_data.get('dateOfBirth') else None,
                        'birth_city': a_data.get('birthPlace', {}).get('city'),
                        'birth_state': a_data.get('birthPlace', {}).get('state'),
                        'birth_country': a_data.get('birthPlace', {}).get('country'),
                        'bats': a_data.get('bats', {}).get('abbreviation'),
                        'throws': a_data.get('throws', {}).get('abbreviation'),
                        'is_active': a_data.get('active', False),
                        'position_id': int(a_data.get('position', {}).get('$ref', '').split('/positions/')[1].split('?')[0]) if 'position' in a_data else None
                    })
                except Exception as e:
                    pass
                    
    last_processed_idx = end_idx
    end_time = time.time()
    print(f"\nBatch finished in {end_time - start_time:.2f} seconds!")
    print(f"Total athletes captured so far: {len(global_athlete_records)}")
    
    if last_processed_idx >= len(athlete_ref_list):
        print("\n--- ALL BATCHES COMPLETE! ---")
        df_athletes = pd.DataFrame(global_athlete_records).drop_duplicates(subset=['athlete_id']).sort_values('athlete_id').reset_index(drop=True)
        df_athletes.set_index('athlete_id', inplace=True, drop=False)
        display(df_athletes.head(5))





Fetching bios for batch: 36500 to 37681...
  ...processed 100/1182 in this batch...
  ...processed 200/1182 in this batch...
  ...processed 300/1182 in this batch...
  ...processed 400/1182 in this batch...
  ...processed 500/1182 in this batch...
  ...processed 600/1182 in this batch...
  ...processed 700/1182 in this batch...
  ...processed 800/1182 in this batch...
  ...processed 900/1182 in this batch...
  ...processed 1000/1182 in this batch...
  ...processed 1100/1182 in this batch...

Batch finished in 3.75 seconds!
Total athletes captured so far: 37639

--- ALL BATCHES COMPLETE! ---


,athlete_id,uid,first_name,last_name,full_name,display_name,weight,height,age,date_of_birth,birth_city,birth_state,birth_country,bats,throws,is_active,position_id
athlete_id,,,,,,,,,,,,,,,,,
1,1,s:1~l:10~a:1,Gary,Alexander,Gary Alexander,Gary Alexander,195.0,74.0,72.0,1953-03-27 08:00:00+00:00,Los Angeles,NaN,United States,R,R,False,3
2,2,s:1~l:10~a:2,Bill,Almon,Bill Almon,Bill Almon,180.0,75.0,73.0,1952-11-21 08:00:00+00:00,Providence,RI,United States,R,R,False,5
3,3,s:1~l:10~a:3,Wendell,Alston,Wendell Alston,Wendell Alston,180.0,72.0,73.0,1952-09-22 07:00:00+00:00,Valhalla,NY,United States,L,R,False,10
4,4,s:1~l:10~a:4,Tony,Armas,Tony Armas,Tony Armas,224.0,73.0,72.0,1953-07-02 07:00:00+00:00,Puerto Piritu,NaN,Venezuela,R,R,False,9
5,5,s:1~l:10~a:5,Alan,Ashby,Alan Ashby,Alan Ashby,195.0,73.0,74.0,1951-07-08 07:00:00+00:00,Long Beach,CA,United States,B,R,False,2


## 6. Events (Games)

Events are the core action of the application. Unlike teams or players, events are best queried by date. We can query the `/events` endpoint using the `dates=YYYY` parameter to get all games for a specific year, which usually totals around 2,500 - 3,000 games per season.

In [18]:
print("Fetching index of all events for the demo period (2019+)...")
event_refs = []

with httpx.Client(limits=httpx.Limits(max_connections=50)) as client:
    # Loop over the years we filtered earlier (2019 to current)
    for year in df_seasons['season_year'].tolist():
        events_url = f"https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/events?dates={year}&limit=1000"
        print(f"Fetching {year}...")
        
        try:
            initial_resp = fetch_data(client, events_url)
            page_count = initial_resp.get('pageCount', 1)
            
            if 'items' in initial_resp:
                event_refs.extend([item['$ref'] for item in initial_resp['items']])
                
            page_urls = [f"{events_url}&page={p}" for p in range(2, page_count + 1)]
            
            with ThreadPoolExecutor(max_workers=20) as executor:
                future_to_url = {executor.submit(fetch_data, client, url): url for url in page_urls}
                for future in as_completed(future_to_url):
                    try:
                        data = future.result()
                        if 'items' in data:
                            event_refs.extend([item['$ref'] for item in data['items']])
                    except:
                        pass
        except Exception as e:
            print(f"Failed to fetch index for {year}: {e}")

event_refs = sorted(list(set(event_refs)))
print(f"\nSuccessfully extracted {len(event_refs)} unique game URLs.")

if 'global_event_records' not in locals(): global_event_records = []
if 'global_event_competitors' not in locals(): global_event_competitors = []
if 'last_event_idx' not in locals(): last_event_idx = 0



Fetching index of all events for the demo period (2019+)...
Fetching 2026...
Fetching 2025...
Fetching 2024...
Fetching 2023...
Fetching 2022...
Fetching 2021...
Fetching 2020...
Fetching 2019...

Successfully extracted 21935 unique game URLs.


## 6B. MANUAL BATCH: Fetch Deep Game Data

This cell fetches the massive `/summary` endpoint for each game. It will extract the core event, the competitors, the batting/pitching boxscores, the play-by-play, and the win probability.

Run this multiple times to batch through all 15,000+ games from 2019-current!

In [34]:



import csv
from io import StringIO

def psql_insert_copy(table, conn, keys, data_iter):
    # Gets a DBAPI connection that can provide a cursor
    dbapi_conn = conn.connection
    with dbapi_conn.cursor() as cur:
        s_buf = StringIO()
        writer = csv.writer(s_buf)
        writer.writerows(data_iter)
        s_buf.seek(0)
        
        columns = ', '.join([f'"{k}"' for k in keys])
        if table.schema:
            table_name = f'"{table.schema}"."{table.name}"'
        else:
            table_name = f'"{table.name}"'
            
        sql = f'COPY {table_name} ({columns}) FROM STDIN WITH CSV'
        cur.copy_expert(sql=sql, file=s_buf)


def safe_int(val):
    if val is None or str(val) == '--' or str(val) == '':
        return None
    try:
        return int(float(str(val)))
    except:
        return None

def fetch_data_safe(client, url, max_retries=3):
    import time
    for attempt in range(max_retries):
        resp = client.get(url)
        if resp.status_code == 403 or resp.status_code == 429:
            wait_time = (2 ** attempt) + 1
            print(f"Rate limited on {url}. Waiting {wait_time}s...")
            time.sleep(wait_time)
            continue
        resp.raise_for_status()
        return resp.json()
    raise Exception(f"Failed to fetch {url}")

if 'global_events' not in locals(): global_events = []
if 'engine' not in locals():
    from sqlalchemy import create_engine
    engine = create_engine("postgresql:///mlb_db")


if 'global_competitors' not in locals(): global_competitors = []
if 'global_batting' not in locals(): global_batting = []
if 'global_pitching' not in locals(): global_pitching = []
if 'global_plays' not in locals(): global_plays = []
if 'global_wp' not in locals(): global_wp = []
if 'last_summary_idx' not in locals(): last_summary_idx = 0
# last_summary_idx = 0 # Hard reset as requested by user

# You can adjust this batch size. 500 is safe to avoid 403s.
BATCH_SIZE = 1500

# If the arrays are already holding more than the expected BATCH_SIZE (meaning a previous write failed),
# we should clear them out so we don't try to write 4500 games and duplicate work.
if len(global_events) > BATCH_SIZE:
    print("Warning: Found leftover data from a failed batch. Clearing memory before starting!")
    global_events = []
    global_competitors = []
    global_batting = []
    global_pitching = []
    global_plays = []
    global_wp = []
start_idx = last_summary_idx
end_idx = min(start_idx + BATCH_SIZE, len(event_refs))

if start_idx >= len(event_refs):
    print("All game summaries have been processed! You can run the DB insertion below.")
else:
    batch_refs = event_refs[start_idx:end_idx]
    
    # We must hit the /summary endpoint, so we swap /events/ for /summary?event=
    # Extract the event ID and build the correct Site API URL
    summary_urls = [f"https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/summary?event={url.split('/events/')[1].split('?')[0]}&region=us&lang=en" for url in batch_refs]
    # Example: https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/summary?event=401569651
    
    print(f"Fetching Deep Game Data for batch: {start_idx} to {end_idx - 1}...")
    start_time = time.time()
    
    with httpx.Client(limits=httpx.Limits(max_connections=150)) as client:
        with ThreadPoolExecutor(max_workers=100) as executor:
            future_to_url = {executor.submit(fetch_data_safe, client, url): url for url in summary_urls}
            
            for i, future in enumerate(as_completed(future_to_url)):
                if i > 0 and i % 100 == 0:
                    print(f"  ...processed {i}/{len(batch_refs)} summaries...")
                try:
                    data = future.result()
                    
                    # Ensure the game is actually completed before saving it
                    status = data.get('header', {}).get('competitions', [{}])[0].get('status', {}).get('type', {}).get('name')
                    if status not in ['STATUS_FINAL', 'STATUS_POSTPONED', 'STATUS_CANCELED']:
                        continue
                    
                    # 1. CORE EVENT
                    event_id = safe_int(data.get('header', {}).get('id'))
                    if not event_id: continue
                    season_year = safe_int(data.get('header', {}).get('season', {}).get('year'))
                    
                    global_events.append({
                        'event_id': event_id,
                        'date': pd.to_datetime(data.get('header', {}).get('competitions', [{}])[0].get('date'), utc=True).replace(tzinfo=None),
                        'name': data.get('header', {}).get('name'),
                        'short_name': data.get('header', {}).get('competitions', [{}])[0].get('shortName'),
                        'season_year': season_year,
                        'attendance': data.get('gameInfo', {}).get('attendance'),
                        'venue_id': int(data.get('gameInfo', {}).get('venue', {}).get('id')) if data.get('gameInfo', {}).get('venue') else None
                    })
                    
                    # 2. COMPETITORS
                    for team in data.get('header', {}).get('competitions', [{}])[0].get('competitors', []):
                        team_id = safe_int(team.get('id'))
                        if not team_id: continue
                        global_competitors.append({
                            'event_competitor_id': f"{event_id}_{team_id}",
                            'event_id': event_id,
                            'team_id': team_id,
                            'season_team_id': f"{season_year}_{team_id}",
                            'home_away': team.get('homeAway'),
                            'winner': team.get('winner'),
                            'score': safe_int(team.get('score'))
                        })
                        
                    # 3. BOXSCORES
                    boxscore = data.get('boxscore', {})
                    for team_box in boxscore.get('players', []):
                        team_id = safe_int(team_box.get('team', {}).get('id'))
                        if not team_id: continue
                        
                        for stat_group in team_box.get('statistics', []):
                            stat_name = stat_group.get('type')
                            labels = stat_group.get('labels', [])
                            
                            for athlete_stats in stat_group.get('athletes', []):
                                athlete_id = safe_int(athlete_stats.get('athlete', {}).get('id'))
                                if not athlete_id: continue
                                stats_list = athlete_stats.get('stats', [])
                                if not stats_list: continue
                                stat_dict = dict(zip(labels, stats_list))
                                
                                if stat_name == "batting":
                                    global_batting.append({
                                        'event_batting_id': f"{event_id}_{athlete_id}",
                                        'event_id': event_id,
                                        'team_id': team_id,
                                        'athlete_id': athlete_id,
                                        'starter': athlete_stats.get('starter', False),
                                        'position_id': int(athlete_stats.get('position', {}).get('id')) if athlete_stats.get('position') else None,
                                        'ab': safe_int(stat_dict.get('AB')),
                                        'r': safe_int(stat_dict.get('R')),
                                        'h': safe_int(stat_dict.get('H')),
                                        'rbi': safe_int(stat_dict.get('RBI')),
                                        'hr': safe_int(stat_dict.get('HR')),
                                        'bb': safe_int(stat_dict.get('BB')),
                                        'k': safe_int(stat_dict.get('K')),
                                        'pitches_faced': safe_int(stat_dict.get('#P'))
                                    })
                                elif stat_name == "pitching":
                                    global_pitching.append({
                                        'event_pitching_id': f"{event_id}_{athlete_id}",
                                        'event_id': event_id,
                                        'team_id': team_id,
                                        'athlete_id': athlete_id,
                                        'starter': athlete_stats.get('starter', False),
                                        'ip': stat_dict.get('IP', stat_dict.get('fullInnings.partInnings')),
                                        'h': safe_int(stat_dict.get('H')),
                                        'r': safe_int(stat_dict.get('R')),
                                        'er': safe_int(stat_dict.get('ER')),
                                        'bb': safe_int(stat_dict.get('BB')),
                                        'k': safe_int(stat_dict.get('K')),
                                        'hr': safe_int(stat_dict.get('HR')),
                                        'pitches': safe_int(stat_dict.get('PC'))
                                    })
                                    
                    # 4. PLAYS
                    for play in data.get('plays', []):
                        global_plays.append({
                            'play_id': play.get('id'),
                            'event_id': event_id,
                            'inning': play.get('period', {}).get('number'),
                            'is_scoring_play': play.get('scoringPlay', False),
                            'score_value': play.get('scoreValue', 0),
                            'away_score': play.get('awayScore'),
                            'home_score': play.get('homeScore'),
                            'play_type_id': safe_int(play.get('type', {}).get('id')),
                            'play_type_text': play.get('type', {}).get('text'),
                            'text': play.get('text'),
                            'pitch_type': play.get('pitchType', {}).get('text'),
                            'pitch_velocity': play.get('pitchVelocity'),
                            'pitch_coordinate_x': play.get('pitchCoordinate', {}).get('x'),
                            'pitch_coordinate_y': play.get('pitchCoordinate', {}).get('y'),
                            'hit_coordinate_x': play.get('hitCoordinate', {}).get('x'),
                            'hit_coordinate_y': play.get('hitCoordinate', {}).get('y')
                        })
                        
                    # 5. WIN PROBABILITY
                    for wp in data.get('winprobability', []):
                        global_wp.append({
                            'play_id': wp.get('playId'),
                            'event_id': event_id,
                            'home_win_percentage': wp.get('homeWinPercentage'),
                            'tie_percentage': wp.get('tiePercentage')
                        })
                        
                except Exception as e:
                    print(f"Error processing {url}: {e}")
                    pass


    last_summary_idx = end_idx
    print(f"Batch finished in {time.time() - start_time:.2f} seconds!")
    print(f"Total processed so far: {last_summary_idx} games")
    
    # --- IMMEDIATE DATABASE WRITE (UPSERT/APPEND) ---

    print(f"Writing {len(global_events)} games to database to free memory...")
    
    if global_events:
        df_e = pd.DataFrame(global_events).drop_duplicates('event_id')
        df_e['venue_id'] = df_e['venue_id'].astype('Int64')
        df_e.to_sql('events', engine, if_exists='append', index=False, method=psql_insert_copy)
        
    if global_competitors:
        df_c = pd.DataFrame(global_competitors).drop_duplicates('event_competitor_id')
        df_c['score'] = df_c['score'].astype('Int64')
        df_c.to_sql('event_competitors', engine, if_exists='append', index=False, method=psql_insert_copy)
        
    if global_batting:
        df_b = pd.DataFrame(global_batting).drop_duplicates('event_batting_id')
        for col in ['position_id', 'ab', 'r', 'h', 'rbi', 'hr', 'bb', 'k', 'pitches_faced']:
            df_b[col] = df_b[col].astype('Int64')
        df_b.to_sql('event_boxscores_batting', engine, if_exists='append', index=False, method=psql_insert_copy)
        
    if global_pitching:
        df_p = pd.DataFrame(global_pitching).drop_duplicates('event_pitching_id')
        for col in ['h', 'r', 'er', 'bb', 'k', 'hr', 'pitches']:
            df_p[col] = df_p[col].astype('Int64')
        df_p.to_sql('event_boxscores_pitching', engine, if_exists='append', index=False, method=psql_insert_copy)
        
    if global_plays:
        df_pl = pd.DataFrame(global_plays).drop_duplicates('play_id')
        for col in ['inning', 'score_value', 'away_score', 'home_score', 'play_type_id']:
            df_pl[col] = df_pl[col].astype('Int64')
        df_pl.to_sql('event_plays', engine, if_exists='append', index=False, method=psql_insert_copy)
        
    if global_wp:
        df_wp = pd.DataFrame(global_wp).drop_duplicates('play_id')
        df_wp.to_sql('event_win_probability', engine, if_exists='append', index=False, method=psql_insert_copy)
        
    # CLEAR MEMORY LISTS FOR NEXT BATCH!

    # CLEAR MEMORY LISTS FOR NEXT BATCH!
    global_events = []
    global_competitors = []
    global_batting = []
    global_pitching = []
    global_plays = []
    global_wp = []
    
    if last_summary_idx >= len(event_refs):
        print("--- ALL SUMMARIES COMPLETE AND SAVED! ---")



















Fetching Deep Game Data for batch: 21000 to 21934...
  ...processed 100/935 summaries...
  ...processed 200/935 summaries...
  ...processed 300/935 summaries...
  ...processed 400/935 summaries...
  ...processed 500/935 summaries...
  ...processed 600/935 summaries...
  ...processed 700/935 summaries...
  ...processed 800/935 summaries...
  ...processed 900/935 summaries...
Batch finished in 7.50 seconds!
Total processed so far: 21935 games
Writing 451 games to database to free memory...
--- ALL SUMMARIES COMPLETE AND SAVED! ---


## 3. Database Ingestion (PostgreSQL)

Now that our core dimension tables (`seasons`, `season_types`, and `season_teams`) are correctly mapped, we'll use SQLAlchemy to push them to a PostgreSQL database. 

*Note: Update the `DATABASE_URL` with your local or cloud Postgres credentials.*

In [36]:
from sqlalchemy import create_engine

# Connect to Local PostgreSQL
DATABASE_URL = "postgresql:///mlb_db"

try:
    engine = create_engine(DATABASE_URL)
    
    print("Writing Dimension Tables to Database...")
    
    if 'df_seasons' in locals():
        print("  -> seasons")
        df_seasons.to_sql('seasons', engine, if_exists='replace', index=False)
        
    if 'df_season_types' in locals():
        print("  -> season_types")
        df_season_types.to_sql('season_types', engine, if_exists='replace', index=False)
        
    if 'df_season_teams' in locals():
        print("  -> season_teams")
        df_season_teams.to_sql('season_teams', engine, if_exists='replace', index=False)
        
    if 'df_groups' in locals():
        print("  -> groups")
        df_groups.to_sql('groups', engine, if_exists='replace', index=False)
        
    if 'df_positions' in locals():
        print("  -> positions")
        df_positions.to_sql('positions', engine, if_exists='replace', index=False)
        
    if 'df_athletes' in locals():
        print("  -> athletes")
        df_athletes.to_sql('athletes', engine, if_exists='replace', index=False)
    
    print("\nAll available dimension tables successfully written!")
    print("Note: The Deep Game Data (events, plays, boxscores) was already automatically saved via COPY append during the batching step.")
    
except Exception as e:
    print(f"\nDatabase Connection or Write Error:\n{e}\n")



Writing Dimension Tables to Database...
  -> seasons
  -> season_types
  -> season_teams
  -> groups
  -> positions

All available dimension tables successfully written!
Note: The Deep Game Data (events, plays, boxscores) was already automatically saved via COPY append during the batching step.
